# Getting Started with OPIE

This notebook walks through OPIE's core concepts:
1. Creating an illustration request
2. Running the illustration
3. Exploring the output
4. Comparing scenarios
5. Modifying assumptions to see the impact

In [ ]:
from datetime import date
from decimal import Decimal

from opie import run_illustration
from opie.core.types import IllustrationRequest
from opie.assumptions.models import ScenarioSet, ULScenarioAssumptions
from opie.core.types import PremiumScheduleEntry

## 1. Build a Simple UL Request

Every illustration starts with a request. The request specifies the product,
policyholder details, and assumptions for each scenario.

In [ ]:
# Define assumptions for 'current' and 'guaranteed' scenarios
coi_table = {age: Decimal("5.00") for age in range(35, 100)}

current = ULScenarioAssumptions(
    crediting_rate_annual=Decimal("0.04"),
    premium_load_pct=Decimal("0.05"),
    monthly_policy_fee=Decimal("10"),
    coi_table=coi_table,
    surrender_charge_schedule={1: Decimal("500"), 60: Decimal("0")},
)

guaranteed = ULScenarioAssumptions(
    crediting_rate_annual=Decimal("0.01"),
    premium_load_pct=Decimal("0.05"),
    monthly_policy_fee=Decimal("10"),
    coi_table=coi_table,
    surrender_charge_schedule={1: Decimal("500"), 60: Decimal("0")},
)

request = IllustrationRequest(
    product_code="simple_ul",
    issue_age=35,
    issue_gender="M",
    risk_class="NT",
    face_amount=Decimal("250000"),
    issue_date=date(2025, 1, 1),
    duration_months=120,
    premium_schedule=[
        PremiumScheduleEntry(start_month=1, end_month=120, amount=Decimal("500")),
    ],
    scenarios=ScenarioSet(current=current, guaranteed=guaranteed),
)

print(f"Product: {request.product_code}")
print(f"Duration: {request.duration_months} months ({request.duration_months // 12} years)")
print(f"Face amount: ${request.face_amount:,.2f}")

## 2. Run the Illustration

OPIE runs both scenarios (current and guaranteed) and returns a result
with one ledger per scenario. All math uses `Decimal` — no floats.

In [ ]:
result = run_illustration(request)

print(f"Request ID: {result.request_id}")
print(f"Calc version: {result.metadata.calc_version}")
print(f"Schema version: {result.metadata.schema_version}")
print(f"Scenarios: {list(result.ledgers.keys())}")

## 3. Explore the Ledger

Each ledger contains monthly rows with all the policy values.

In [ ]:
current_ledger = result.ledgers["current"]

print(f"Rows: {len(current_ledger.rows)}")
print()

# Show key values at years 1, 5, and 10
for year in [1, 5, 10]:
    row = current_ledger.rows[year * 12 - 1]  # month index
    print(f"Year {year:2d} (month {row.t:3d}): "
          f"AV=${row.account_value_eop:>10,.2f}  "
          f"CSV=${row.cash_surrender_value:>10,.2f}  "
          f"DB=${row.death_benefit:>12,.2f}  "
          f"Cum Premium=${row.cumulative_premium:>10,.2f}")

## 4. Compare Scenarios

The guaranteed scenario uses a lower crediting rate — see how it
affects account values over time.

In [ ]:
guaranteed_ledger = result.ledgers["guaranteed"]

print(f"{'Year':>4}  {'Current AV':>12}  {'Guaranteed AV':>14}  {'Difference':>12}")
print("-" * 50)

for year in range(1, 11):
    c_row = current_ledger.rows[year * 12 - 1]
    g_row = guaranteed_ledger.rows[year * 12 - 1]
    diff = c_row.account_value_eop - g_row.account_value_eop
    print(f"{year:4d}  ${c_row.account_value_eop:>11,.2f}  ${g_row.account_value_eop:>13,.2f}  ${diff:>11,.2f}")

## 5. Modify Assumptions

Change the crediting rate to see the impact. This is the power of
a deterministic illustration engine — you can run scenarios instantly.

In [ ]:
# Try a higher crediting rate
high_rate = current.model_copy(update={"crediting_rate_annual": Decimal("0.06")})
high_request = request.model_copy(
    update={"scenarios": ScenarioSet(current=high_rate, guaranteed=guaranteed)}
)
high_result = run_illustration(high_request)

print("Year 10 account values at different crediting rates:")
print(f"  4% (original): ${current_ledger.rows[119].account_value_eop:>12,.2f}")
print(f"  6% (higher):   ${high_result.ledgers['current'].rows[119].account_value_eop:>12,.2f}")
print(f"  1% (guaranteed): ${guaranteed_ledger.rows[119].account_value_eop:>12,.2f}")

## Next Steps

- Try different products: `level_term`, `wl_nonpar`, `annuity_deferred`, `annuity_spia`
- Enable premium solve: add a `solve` config to find the minimum premium
- Add riders, withdrawals, or loans
- Use the CLI: `opie illustrate --in request.json --out result.json`
- Explore the API: `uvicorn opie.api.app:app --reload`

See the [README](../README.md) and [docs/](../docs/) for full documentation.